In [1]:
# =========================
# STEP 1: Install & Imports
# =========================

!pip install torch-geometric -q

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_undirected
from torch_geometric.transforms import NormalizeFeatures

from collections import defaultdict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# Load Cora
# =========================

dataset = Planetoid(
    root='/kaggle/working/Cora',
    name='Cora',
    transform=NormalizeFeatures()
)

data = dataset[0]   # ⚠️ KEEP ON CPU for DOSAGE (important)

print("Number of nodes:", data.num_nodes)
print("Number of edges:", data.num_edges)
print("Number of features:", dataset.num_node_features)
print("Number of classes:", dataset.num_classes)

# =========================
# Graph Preprocessing (CRITICAL for DOSAGE)
# =========================

# Convert to undirected graph (paper assumes simple graph G = (V, E'))
edge_index = to_undirected(data.edge_index)

# Build adjacency list (needed for density, induced subgraph, diameter)
adj = defaultdict(set)

for u, v in edge_index.t().tolist():
    adj[u].add(v)
    adj[v].add(u)

# Optional: store node set
V = set(range(data.num_nodes))

print("Graph preprocessing done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.8 MB/s eta 0:00:00a 0:00:01
Using device: cpu


Processing...


Number of nodes: 2708
Number of edges: 10556
Number of features: 1433
Number of classes: 7
Graph preprocessing done.


Done!


In [15]:
# =====================================
# STEP 2: Define HGNN Layer and Model
# =====================================

class HGNNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super(HGNNLayer, self).__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x, G):
        x = self.linear(x)
        x = torch.sparse.mm(G, x)
        return x


class HGNN(nn.Module):
    def __init__(self, in_features, hidden_dim, num_classes, dropout=0.5):
        super(HGNN, self).__init__()

        self.h1 = HGNNLayer(in_features, hidden_dim)
        self.h2 = HGNNLayer(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, G):
        x = self.h1(x, G)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.h2(x, G)
        return x

# Hyperedge Modelling Using Dosage

In [2]:
import networkx as nx
from torch_geometric.utils import to_undirected

# Convert to undirected graph
edge_index = to_undirected(data.edge_index)

G_nx = nx.Graph()
G_nx.add_nodes_from(range(data.num_nodes))  # IMPORTANT: keep all nodes
G_nx.add_edges_from(edge_index.t().cpu().numpy().tolist())

print("Graph:", G_nx.number_of_nodes(), "nodes,", G_nx.number_of_edges(), "edges")

Graph: 2708 nodes, 5278 edges


In [3]:
# ---------------------------
# Global counters (TOTAL)
# ---------------------------
GLOBAL_STATS = {
    "total_subsets": 0,
    "valid_subsets": 0,
    "diameter_checks": 0,
    "density_calls": 0,
    "distance_calls": 0,
    "objective_calls": 0
}

# ---------------------------
# Local counters (per hyperedge)
# ---------------------------
LOCAL_STATS = {}

def RESET_LOCAL_STATS():
    global LOCAL_STATS
    LOCAL_STATS = {
        "subsets": 0,
        "valid_subsets": 0,
        "diameter_checks": 0,
        "density_calls": 0,
        "distance_calls": 0,
        "objective_calls": 0
    }

In [4]:
# ---------------------------
# Density
# ---------------------------
def DENSITY(G):
    # As per paper: |E(S)| / |S|
    GLOBAL_STATS["density_calls"] += 1
    LOCAL_STATS["density_calls"] += 1

    if G.number_of_nodes() == 0:
        return 0
    return G.number_of_edges() / G.number_of_nodes()

In [5]:
# ---------------------------
# Distance between subgraphs
# ---------------------------
def DISTANCE(pair1, pair2):
    """
    CHANGE:
    We pass (graph, node_set) to avoid recomputing nodes.
    Logic is unchanged.
    """

    GLOBAL_STATS["distance_calls"] += 1
    LOCAL_STATS["distance_calls"] += 1

    G1, U = pair1
    G2, Z = pair2

    if len(U) == 0 or len(Z) == 0:
        return 2

    if U == Z:
        return 0

    intersection = len(U.intersection(Z))
    return 2 - (intersection ** 2) / (len(U) * len(Z))

In [6]:
# ---------------------------
# Objective function
# ---------------------------
def OBJECTIVE(W, lam):
    """
    W stores (graph, node_set)
    """

    GLOBAL_STATS["objective_calls"] += 1
    LOCAL_STATS["objective_calls"] += 1

    total_density = sum(DENSITY(Gs) for Gs, _ in W)

    total_distance = 0
    for i in range(len(W)):
        for j in range(i + 1, len(W)):
            total_distance += DISTANCE(W[i], W[j])

    return total_density + lam * total_distance

In [12]:
# ---------------------------
# Densest Subgraph (peeling)
# ---------------------------
def DENSESTSUBGRAPH(G, alpha, beta, delta):
    
    G_current = G.copy()
    G_best = None
    best_density = 0
    
    while G_current.number_of_nodes() > 0:
        
        n = G_current.number_of_nodes()
        
        # Stop if below minimum size
        if n < alpha:
            break
        
        # ---------------------------
        # Diameter constraint (FIXED)
        # ---------------------------
        if nx.is_connected(G_current):
            diam = nx.diameter(G_current)
        else:
            # Use largest connected component instead of rejecting
            largest_cc = max(nx.connected_components(G_current), key=len)
            G_cc = G_current.subgraph(largest_cc)
            
            if G_cc.number_of_nodes() > 1:
                diam = nx.diameter(G_cc)
            else:
                diam = 0
        
        # ---------------------------
        # Check constraints + update best
        # ---------------------------
        if diam <= delta and alpha <= n <= beta:
            d = DENSITY(G_current)
            if d > best_density:
                best_density = d
                G_best = G_current.copy()
        
        # ---------------------------
        # Goldberg peeling (remove min-degree nodes)
        # ---------------------------
        degrees = dict(G_current.degree())
        min_deg = min(degrees.values())
        remove_nodes = [v for v, d in degrees.items() if d == min_deg]
        G_current.remove_nodes_from(remove_nodes)
    
    # ---------------------------
    # Return with node set
    # ---------------------------
    if G_best is None:
        return None
    
    return (G_best, set(G_best.nodes()))

In [26]:
!pip install tqdm -q

In [28]:
from tqdm import tqdm
import itertools

def DENSESTDISTINCTSUBGRAPH(G, W, lam, alpha, beta, delta):
    
    best_candidate = None
    best_obj = -1
    
    V = list(G.nodes())
    n = len(V)
    
    # ---------------------------
    # Compute total subsets (for tqdm)
    # ---------------------------
    total_subsets = sum(math.comb(n, k) for k in range(alpha, beta + 1))
    
    # tqdm progress bar
    pbar = tqdm(total=total_subsets, desc="Enumerating subsets", leave=False)
    
    for size in range(alpha, beta + 1):
        
        for subset in itertools.combinations(V, size):
            
            # update progress bar ONLY for subset enumeration
            pbar.update(1)
            if LOCAL_STATS["subsets"] % 1000 == 0:   # update every 1000 steps (avoid slowdown)
                pbar.set_postfix({
                "valid": LOCAL_STATS["valid_subsets"]
        })
            # ---------------------------
            # Stats (optional, keep if you want)
            # ---------------------------
            GLOBAL_STATS["total_subsets"] += 1
            LOCAL_STATS["subsets"] += 1
            
            S = set(subset)
            G_sub = G.subgraph(S).copy()
            
            # ---------------------------
            # Diameter check (FIXED)
            # ---------------------------
            GLOBAL_STATS["diameter_checks"] += 1
            LOCAL_STATS["diameter_checks"] += 1
            
            if nx.is_connected(G_sub):
                diam = nx.diameter(G_sub)
            else:
                largest_cc = max(nx.connected_components(G_sub), key=len)
                G_cc = G_sub.subgraph(largest_cc)
                
                if G_cc.number_of_nodes() > 1:
                    diam = nx.diameter(G_cc)
                else:
                    diam = 0
            
            if diam > delta:
                continue
            
            GLOBAL_STATS["valid_subsets"] += 1
            LOCAL_STATS["valid_subsets"] += 1
            
            candidate = (G_sub, S)
            
            # Distinctness
            distinct = True
            for prev in W:
                if DISTANCE(candidate, prev) == 0:
                    distinct = False
                    break
            
            if not distinct:
                continue
            
            # Objective
            W_temp = W + [candidate]
            obj = OBJECTIVE(W_temp, lam)
            
            if obj > best_obj:
                best_obj = obj
                best_candidate = candidate
    
    pbar.close()
    
    return best_candidate

In [18]:
import math

def DOSAGE(G, k, lam, alpha, beta):
    
    if nx.is_connected(G):
        delta = 2 * nx.average_shortest_path_length(G)
    else:
        delta = math.log2(G.number_of_nodes())
    
    print(f"Computed delta: {delta:.3f}")
    
    # First hyperedge
    G_initial = DENSESTSUBGRAPH(G, alpha, beta, delta)
    
    if G_initial is None:
        W = []
    else:
        G0, S0 = G_initial
        W = [G_initial]
        
        obj = OBJECTIVE(W, lam)
        
        print("\n=== Hyperedge 1 ===")
        print(f"Size       : {len(S0)}")
        print(f"Density    : {DENSITY(G0):.4f}")
        print(f"Objective  : {obj:.4f}")
    
    # Remaining hyperedges
    while len(W) < k:
        
        print(f"\n--- Searching Hyperedge {len(W)+1} ---")
        
        RESET_LOCAL_STATS()
        
        G_next = DENSESTDISTINCTSUBGRAPH(G, W, lam, alpha, beta, delta)
        
        if G_next is None:
            print("No more valid subgraphs found.")
            break
        
        Gg, Sg = G_next
        
        distances = [DISTANCE(G_next, prev) for prev in W]
        obj = OBJECTIVE(W + [G_next], lam)
        
        W.append(G_next)
        
        print(f"\n=== Hyperedge {len(W)} ===")
        print(f"Size       : {len(Sg)}")
        print(f"Density    : {DENSITY(Gg):.4f}")
        print(f"Distances  : {[round(d,4) for d in distances]}")
        print(f"Objective  : {obj:.4f}")
        
        print("\n[Stats for this Hyperedge]")
        for k_stat, v in LOCAL_STATS.items():
            print(f"{k_stat:20s}: {v}")
    
    print("\n===== TOTAL STATS =====")
    for k_stat, v in GLOBAL_STATS.items():
        print(f"{k_stat:20s}: {v}")
    
    return W

In [ ]:
# SMALL GRAPH ONLY
subset_nodes = list(range(50))
G_small = G_nx.subgraph(subset_nodes).copy()

W = DOSAGE(
    G_small,
    k=5,
    lam=2,
    alpha=3,
    beta=10
)

print("\nTotal hyperedges:", len(W))

covered = set()
for _, S in W:
    covered.update(S)

print("Coverage:", len(covered))

if len(W) > 0:
    avg_size = sum(len(S) for _, S in W) / len(W)
    print("Avg size:", avg_size)
else:
    print("Avg size: N/A")

Computed delta: 5.644

=== Hyperedge 1 ===
Size       : 6
Density    : 0.5000
Objective  : 0.5000

--- Searching Hyperedge 2 ---


Enumerating subsets:   0%|          | 18753914/13432734280 [29:19<354:43:09, 10504.38it/s, valid=1.88e+7]